# Heart Sound Analysis -- Automated Benchmark & Audit Pipeline
### Leakage-Aware Evaluation, 5 Architectural Ablations, Explanation Audits & Paper Deliverables

**Instructions:**
1. Go to **Runtime -> Change runtime type** and ensure **T4 GPU** is selected.
2. Click **Runtime -> Run all** (or press `Ctrl + F9`).
3. The notebook will automatically execute the entire 24-experiment benchmark, audit explanation faithfulness, compile all LaTeX tables and macros, and download the results.


## 0. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# Install required Python dependencies
%pip install -q librosa scipy numpy pandas torch torchvision tqdm shap h5py pytest

## 1. Repository Setup & Drive Mount (Optional)

In [ ]:
import os
from pathlib import Path

# Check if running in Google Drive or fresh Colab instance
if Path("/content/drive/MyDrive").exists():
    print("Google Drive is already mounted.")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted, using local /content environment.")

# Locate project directory
possible_paths = [
    Path("/content/drive/MyDrive/CSE/College/RMS"),
    Path("/content/drive/MyDrive/RMS"),
    Path("/content/heart-sound-analysis"),
    Path.cwd()
]

repo_dir = None
for p in possible_paths:
    if (p / "Review 3").exists() and (p / "physionet_2016").exists():
        repo_dir = p
        break

if repo_dir is None:
    print("Cloning repository into /content/heart-sound-analysis...")
    !git clone https://github.com/urvagandhi/heart-sound-analysis.git /content/heart-sound-analysis
    repo_dir = Path("/content/heart-sound-analysis")

os.chdir(str(repo_dir))
print(f"Active working directory: {Path.cwd()}")

## 2. Dataset Verification & Extraction

In [ ]:
import zipfile
data_dir = Path("physionet_2016")
raw_dir = data_dir / "raw"
spec_dir = data_dir / "spectrograms"
zip_file = Path("physionet_2016.zip")

# Extract if zip exists and spectrograms not yet unpacked
if zip_file.exists() and (not spec_dir.exists() or len(list(spec_dir.glob("*.npy"))) < 100):
    print("Extracting physionet_2016.zip into physionet_2016/...")
    with zipfile.ZipFile(zip_file, "r") as z:
        for m in z.infolist():
            rel_name = m.filename.split("physionet_2016/")[-1]
            if not rel_name:
                continue
            target = data_dir / rel_name
            if m.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, "wb") as dst:
                    dst.write(src.read())
    print("Extraction complete!")

print(f"Spectrograms available: {len(list(spec_dir.glob('*.npy')))} / 3541")
print(f"Metadata CSV exists: {(data_dir / 'metadata.csv').exists()}")

## 3. Generate Splits & Run Duplicate Leakage Audit

In [ ]:
# Generate deduplicated (3 seeds) and LODO splits
!python "Review 3/make_splits.py" --mode dedup --seeds 42 43 44
!python "Review 3/make_splits.py" --mode lodo --holdouts a b e f --seed 42

# Check twin duplicate performance vs unique test accuracy
!python "Review 3/verify_duplicates.py"

## 4. Run All 24 Benchmark Experiments Sequentially

In [ ]:
# Dry-run verification (24 planned runs)
!python "Review 3/run_all.py" --dry-run

# Execute full 24-run benchmark (resume-safe: skips already completed runs)
!python "Review 3/run_all.py"

## 5. Run Population-Level Explanation Audit Suite

In [ ]:
# Execute audit on primary model: dedup_se_mha_seed42
!python "Review 3/audit/run_audit.py" --tag dedup_se_mha_seed42

## 6. Aggregate Deliverables & Compile LaTeX Paper

In [ ]:
# Aggregate all JSON artifacts into LaTeX tables, macros, and findings
!python "Review 3/aggregate_results.py"

# Run unit tests to verify consistency
!python -m pytest -q

# Compile PDF using pdflatex
!apt-get update -qq && apt-get install -y -qq texlive-latex-base texlive-fonts-recommended texlive-fonts-extra texlive-latex-extra
%cd "Paper Writing"
!pdflatex -interaction=nonstopmode paper.tex
!bibtex paper
!pdflatex -interaction=nonstopmode paper.tex
!pdflatex -interaction=nonstopmode paper.tex
%cd ..

## 7. Download Results Archive

In [ ]:
from google.colab import files

# Create zip of all experimental results and the compiled paper PDF
output_archive = "benchmark_results_and_paper.zip"
!zip -q -r benchmark_results_and_paper.zip results "Paper Writing/paper.pdf" "Paper Writing/generated"
print(f"Created archive: {output_archive}")
files.download(output_archive)